# Notebook 10 — Production Catalogue

Goal:

Create a catalogue of Broadway productions from the modern Broadway era
(starting with Show Boat in 1927).

Pipeline:

Season catalogue
        ↓
Production catalogue
        ↓
Production pages
        ↓
Cast extraction
        ↓
Production-performer table
        ↓
Performer network

Output:

data/processed/production_catalogue.csv

In [22]:
import pandas as pd
import requests
import re
import time

from bs4 import BeautifulSoup
from pathlib import Path

In [23]:
ROOT = Path("..")

DATA_PROCESSED = ROOT / "data" / "processed"
DATA_RAW = ROOT / "data" / "raw"

DATA_PROCESSED

PosixPath('../data/processed')

In [24]:
season_df = pd.read_csv(
    DATA_PROCESSED / "season_catalogue.csv"
)

season_df.head()

,season_id,season_label,start_year
0,1001,1899-1900,1899
1,1002,1900-1901,1900
2,1003,1901-1902,1901
3,1004,1902-1903,1902
4,1005,1903-1904,1903


In [25]:
season_df["start_year"] = (
    season_df["season_label"]
    .str.extract(r"(\d{4})")
    .astype(int)
)

season_df.head()

,season_id,season_label,start_year
0,1001,1899-1900,1899
1,1002,1900-1901,1900
2,1003,1901-1902,1901
3,1004,1902-1903,1902
4,1005,1903-1904,1903


In [26]:
season_df["start_year"].describe()
season_df.sort_values("start_year").head()

,season_id,season_label,start_year
101,1102,1750-1751,1750
249,1267,1751-1752,1751
102,1103,1752-1753,1752
103,1104,1753-1754,1753
104,1105,1754-1755,1754


In [27]:
modern_seasons = season_df[
    season_df["start_year"] >= 1927
].copy()

modern_seasons.head()

,season_id,season_label,start_year
28,1029,1927-1928,1927
29,1030,1928-1929,1928
30,1031,1929-1930,1929
31,1032,1930-1931,1930
32,1033,1931-1932,1931


In [28]:
modern_seasons.shape

(100, 3)

In [29]:
def get_season_productions(season_id):
    """
    Extract all Broadway productions listed for a season.
    
    Returns:
        production_id
        title
        url
        season_id
    """

    url = f"https://www.ibdb.com/season/{season_id}"

    response = requests.get(url)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    rows = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        match = re.search(
            r"/broadway-production/.*-(\d+)$",
            href
        )

        if match:
            rows.append({
                "production_id": match.group(1),
                "title": a.get_text(strip=True),
                "url": "https://www.ibdb.com" + href,
                "season_id": season_id
            })

    df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset="production_id")
        .reset_index(drop=True)
    )

    if len(df) > 0:
        assert df["production_id"].is_unique

    return df

In [30]:
test = get_season_productions(
    modern_seasons.iloc[0]["season_id"]
)

test.head()

test.shape

(327, 4)

In [31]:
production_frames = []

for i, season_id in enumerate(modern_seasons["season_id"]):

    try:

        df = get_season_productions(season_id)

        production_frames.append(df)

        print(
            f"{i+1}/{len(modern_seasons)} "
            f"season {season_id}: "
            f"{len(df)} productions"
        )

    except Exception as e:

        print(
            f"FAILED season {season_id}: {e}"
        )

    time.sleep(1)

1/100 season 1029: 327 productions
2/100 season 1030: 269 productions
3/100 season 1031: 302 productions
4/100 season 1032: 226 productions
5/100 season 1033: 235 productions
6/100 season 1034: 202 productions
7/100 season 1035: 171 productions
8/100 season 1036: 217 productions
9/100 season 1037: 179 productions
10/100 season 1038: 187 productions
11/100 season 1039: 146 productions
12/100 season 1040: 134 productions
13/100 season 1041: 103 productions
14/100 season 1042: 91 productions
15/100 season 1043: 106 productions
16/100 season 1044: 94 productions
17/100 season 1045: 137 productions
18/100 season 1046: 129 productions
19/100 season 1047: 128 productions
20/100 season 1048: 142 productions
21/100 season 1049: 132 productions
22/100 season 1050: 96 productions
23/100 season 1051: 80 productions
24/100 season 1052: 111 productions
25/100 season 1053: 94 productions
26/100 season 1054: 83 productions
27/100 season 1055: 85 productions
28/100 season 1056: 79 productions
29/100 se

In [32]:
production_catalogue = pd.concat(
    production_frames,
    ignore_index=True
)

production_catalogue.head()

,production_id,title,url,season_id
0,10336,Bottomland,https://www.ibdb.com/broadway-production/botto...,1029
1,10337,Padlocks of 1927,https://www.ibdb.com/broadway-production/padlo...,1029
2,10338,Madame X,https://www.ibdb.com/broadway-production/madam...,1029
3,444423,Africana [1927],https://www.ibdb.com/broadway-production/afric...,1029
4,10340,Rang Tang,https://www.ibdb.com/broadway-production/rang-...,1029


In [33]:
production_catalogue = production_catalogue.merge(
    modern_seasons[
        [
            "season_id",
            "season_label",
            "start_year"
        ]
    ],
    on="season_id",
    how="left"
)

production_catalogue.head()

,production_id,title,url,season_id,season_label,start_year
0,10336,Bottomland,https://www.ibdb.com/broadway-production/botto...,1029,1927-1928,1927
1,10337,Padlocks of 1927,https://www.ibdb.com/broadway-production/padlo...,1029,1927-1928,1927
2,10338,Madame X,https://www.ibdb.com/broadway-production/madam...,1029,1927-1928,1927
3,444423,Africana [1927],https://www.ibdb.com/broadway-production/afric...,1029,1927-1928,1927
4,10340,Rang Tang,https://www.ibdb.com/broadway-production/rang-...,1029,1927-1928,1927


In [34]:
production_catalogue.shape
production_catalogue.isna().sum()
production_catalogue["production_id"].duplicated().sum()

np.int64(1580)

In [35]:
production_catalogue.to_csv(
    DATA_PROCESSED / "production_catalogue.csv",
    index=False
)